# Starting11 Batch Scraper

## Tournament Discovery & Batch Processing

This notebook helps you:
1. **Discover matches** from a World Cup tournament page
2. **Batch scrape** all lineups for a tournament
3. **Extract unique countries** for the dropdown list

### Tournament URLs:
- WC 2022: `https://www.transfermarkt.com/weltmeisterschaft-2022/gesamtspielplan/pokalwettbewerb/WM22/saison_id/2021`
- WC 2018: `https://www.transfermarkt.com/weltmeisterschaft-2018/gesamtspielplan/pokalwettbewerb/WM18/saison_id/2017`
- WC 2014: `https://www.transfermarkt.com/weltmeisterschaft-2014/gesamtspielplan/pokalwettbewerb/WM14/saison_id/2013`
- WC 2010: `https://www.transfermarkt.com/weltmeisterschaft-2010/gesamtspielplan/pokalwettbewerb/WM10/saison_id/2009`
- WC 2006: `https://www.transfermarkt.com/weltmeisterschaft-2006/gesamtspielplan/pokalwettbewerb/WM06/saison_id/2005`

In [28]:
# Cell: ALL WORLD CUPS CONFIGURATION (1998-2022)
# This defines all tournaments to scrape

ALL_TOURNAMENTS = [
    {
        "code": "wc2022",
        "name": "World Cup 2022",
        "url": "https://www.transfermarkt.com/weltmeisterschaft-2022/gesamtspielplan/pokalwettbewerb/WM22/saison_id/2021"
    },
    {
        "code": "wc2018",
        "name": "World Cup 2018", 
        "url": "https://www.transfermarkt.com/weltmeisterschaft-2018/gesamtspielplan/pokalwettbewerb/WM18/saison_id/2017"
    },
    {
        "code": "wc2014",
        "name": "World Cup 2014",
        "url": "https://www.transfermarkt.com/weltmeisterschaft-2014/gesamtspielplan/pokalwettbewerb/WM14/saison_id/2013"
    },
    {
        "code": "wc2010",
        "name": "World Cup 2010",
        "url": "https://www.transfermarkt.com/weltmeisterschaft-2010/gesamtspielplan/pokalwettbewerb/WM10/saison_id/2009"
    },
    {
        "code": "wc2006",
        "name": "World Cup 2006",
        "url": "https://www.transfermarkt.com/weltmeisterschaft-2006/gesamtspielplan/pokalwettbewerb/WM06/saison_id/2005"
    },
    {
        "code": "wc2002",
        "name": "World Cup 2002",
        "url": "https://www.transfermarkt.com/weltmeisterschaft-2002/gesamtspielplan/pokalwettbewerb/WM02/saison_id/2001"
    },
    {
        "code": "wc1998",
        "name": "World Cup 1998",
        "url": "https://www.transfermarkt.com/weltmeisterschaft-1998/gesamtspielplan/pokalwettbewerb/WM98/saison_id/1997"
    },
]

print(f"Configured {len(ALL_TOURNAMENTS)} World Cups:")
for t in ALL_TOURNAMENTS:
    print(f"  - {t['name']} ({t['code']})")

# For single tournament testing, set these:
TOURNAMENT_URL = ALL_TOURNAMENTS[1]["url"]  # Change index to select tournament
TOURNAMENT_CODE = ALL_TOURNAMENTS[1]["code"]
TOURNAMENT_NAME = ALL_TOURNAMENTS[1]["name"]

print(f"Fetching tournament schedule: {TOURNAMENT_NAME}")
print(f"URL: {TOURNAMENT_URL}")
print("-" * 60)

soup = fetch_page(TOURNAMENT_URL, delay=False)

if soup:
    # Save raw HTML for inspection
    with open(f"/tmp/{TOURNAMENT_CODE}_schedule.html", "w") as f:
        f.write(str(soup))
    print(f"✅ Saved raw HTML to /tmp/{TOURNAMENT_CODE}_schedule.html")
    print("   Open this file in a browser to inspect the structure")
else:
    print("❌ Failed to fetch page")

Fetching tournament schedule: World Cup 2018
URL: https://www.transfermarkt.com/weltmeisterschaft-2018/gesamtspielplan/pokalwettbewerb/WM18/saison_id/2017
------------------------------------------------------------
  Fetching: https://www.transfermarkt.com/weltmeisterschaft-2018/gesamtspielplan/pokalwettbewerb/WM18/saison_id/2017
✅ Saved raw HTML to /tmp/wc2018_schedule.html
   Open this file in a browser to inspect the structure


In [31]:
# Cell: Find all match links using the ergebnis-link class
# This is the most reliable selector - it's the score link for each match

# Find all score links with class "ergebnis-link"
match_links = soup.find_all('a', class_='ergebnis-link')
print(f"Found {len(match_links)} ergebnis-link elements")

# Extract match IDs from the id attribute (most direct) or href
match_ids = []
for link in match_links:
    # The id attribute contains the match ID directly
    match_id = link.get('id')
    if match_id and match_id.isdigit():
        match_ids.append(match_id)

# Remove duplicates while preserving order
match_ids = list(dict.fromkeys(match_ids))

print(f"\nUnique match IDs found: {len(match_ids)}")
for mid in match_ids[:10]:
    print(f"  {mid}")
if len(match_ids) > 10:
    print(f"  ... and {len(match_ids) - 10} more")

Found 64 ergebnis-link elements

Unique match IDs found: 64
  2977683
  2977684
  2977685
  2977686
  2977687
  2977688
  2977689
  2977690
  2977691
  2977692
  ... and 54 more


In [32]:
# Cell: Extract match details (teams) from each ergebnis-link row
# Each score link sits in a row with team names on either side

discovered_matches = []

# Find all score links
score_links = soup.find_all('a', class_='ergebnis-link')
print(f"Processing {len(score_links)} match links...")

for link in score_links:
    match_id = link.get('id')
    if not match_id or not match_id.isdigit():
        continue
    
    # Skip if we already have this match
    if any(m['match_id'] == match_id for m in discovered_matches):
        continue
    
    # Find the parent row (tr) containing this score
    row = link.find_parent('tr')
    if not row:
        continue
    
    # Find team names - look for links to /startseite/verein/ with title attribute
    team_links = row.find_all('a', href=re.compile(r'/startseite/verein/\d+'))
    teams = []
    for tl in team_links:
        title = tl.get('title', '')
        if title and title not in teams:
            teams.append(title)
    
    if len(teams) >= 2:
        discovered_matches.append({
            'match_id': match_id,
            'team1': teams[0],
            'team2': teams[1],
        })

print(f"\n✅ Discovered {len(discovered_matches)} matches with team info:")
for m in discovered_matches[:15]:
    print(f"  {m['match_id']}: {m['team1']} vs {m['team2']}")
if len(discovered_matches) > 15:
    print(f"  ... and {len(discovered_matches) - 15} more")

Processing 64 match links...

✅ Discovered 64 matches with team info:
  2977683: Russia vs Saudi Arabia
  2977684: Egypt vs Uruguay
  2977685: Russia vs Egypt
  2977686: Uruguay vs Saudi Arabia
  2977687: Uruguay vs Russia
  2977688: Saudi Arabia vs Egypt
  2977689: Morocco vs Iran
  2977690: Portugal vs Spain
  2977691: Portugal vs Morocco
  2977692: Iran vs Spain
  2977693: Iran vs Portugal
  2977694: Spain vs Morocco
  2977695: France vs Australia
  2977696: Peru vs Denmark
  2977698: Denmark vs Australia
  ... and 49 more


In [ ]:
# Cell: Batch scrape all discovered matches
# WARNING: This will take a while! Each match requires ~8 seconds (2 page fetches)

OUTPUT_DIR = f"../quizzes/starting11/{TOURNAMENT_CODE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SKIP_EXISTING = True  # Set to False to re-scrape everything

success_count = 0
skipped_count = 0
failed_matches = []
all_countries = set()

print(f"Scraping {len(discovered_matches)} matches for {TOURNAMENT_NAME}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Skip existing: {SKIP_EXISTING}")
print(f"Estimated time: ~{len(discovered_matches) * 8 / 60:.1f} minutes")
print("=" * 60)

for i, match in enumerate(discovered_matches):
    match_id = match['match_id']
    team1 = match['team1']
    team2 = match['team2']
    
    print(f"\n[{i+1}/{len(discovered_matches)}] {team1} vs {team2} (ID: {match_id})")
    
    # Build filenames
    stage_slug = "match"  # Generic stage
    file1 = f"{stage_slug}_{match_id}_{team1.lower().replace(' ', '_')}.json"
    file2 = f"{stage_slug}_{match_id}_{team2.lower().replace(' ', '_')}.json"
    
    if SKIP_EXISTING and os.path.exists(os.path.join(OUTPUT_DIR, file1)):
        print(f"  ⏭️ Skipped (file exists)")
        skipped_count += 1
        continue
    
    # Build URLs - use correct pattern with /spielbericht/index/ prefix
    slug = f"{team1.lower().replace(' ', '-')}_{team2.lower().replace(' ', '-')}"
    lineup_url = f"/spielbericht/aufstellung/spielbericht/{match_id}"
    index_url = f"/spielbericht/index/spielbericht/{match_id}"
    
    try:
        # Scrape lineup
        lineup_data = scrape_lineup_page(lineup_url)
        if len(lineup_data) < 22:
            print(f"  ❌ Not enough players ({len(lineup_data)})")
            failed_matches.append((team1, team2, f"Not enough players ({len(lineup_data)})"))
            continue
        
        # Scrape positions
        positions_data = scrape_positions_page(index_url)
        if len(positions_data) < 2:
            print(f"  ❌ Missing positions ({len(positions_data)} teams)")
            failed_matches.append((team1, team2, "Missing positions"))
            continue
        
        # Merge and create quizzes
        team1_players = lineup_data[:11]
        team2_players = lineup_data[11:22]
        
        team1_merged = merge_player_data(team1_players, positions_data[0])
        team2_merged = merge_player_data(team2_players, positions_data[1])
        
        # Create quiz JSON
        quiz1 = create_quiz(team1, team1[:3].upper(), team2, team1_merged, TOURNAMENT_NAME, "Match", "")
        quiz2 = create_quiz(team2, team2[:3].upper(), team1, team2_merged, TOURNAMENT_NAME, "Match", "")
        
        # Save files
        with open(os.path.join(OUTPUT_DIR, file1), 'w') as f:
            json.dump(quiz1, f, indent=2)
        with open(os.path.join(OUTPUT_DIR, file2), 'w') as f:
            json.dump(quiz2, f, indent=2)
        
        print(f"  ✅ Saved: {file1}")
        success_count += 1
        all_countries.add(team1)
        all_countries.add(team2)
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        failed_matches.append((team1, team2, str(e)))

print("\n" + "=" * 60)
print(f"COMPLETE!")
print(f"  ✅ Scraped: {success_count} matches ({success_count * 2} quizzes)")
print(f"  ⏭️ Skipped: {skipped_count} matches")
print(f"  ❌ Failed: {len(failed_matches)} matches")
print(f"\nCountries found: {sorted(all_countries)}")

if failed_matches:
    print(f"\nFailed matches:")
    for t1, t2, reason in failed_matches:
        print(f"  - {t1} vs {t2}: {reason}")

In [ ]:
# ============================================================
# MASTER BATCH SCRAPER - ALL WORLD CUPS (1998-2022)
# ============================================================
# This cell scrapes ALL tournaments in one run
# Estimated time: ~45-60 minutes for all 7 World Cups

SKIP_EXISTING = True  # Skip matches we already have

# Track overall stats
grand_total_success = 0
grand_total_skipped = 0
grand_total_failed = 0
all_failed_matches = []
all_countries = set()

print("=" * 70)
print("STARTING11 WORLD CUP BATCH SCRAPER (1998-2022)")
print("=" * 70)
print(f"Tournaments to scrape: {len(ALL_TOURNAMENTS)}")
print(f"Skip existing: {SKIP_EXISTING}")
print("=" * 70)

for tournament in ALL_TOURNAMENTS:
    t_code = tournament["code"]
    t_name = tournament["name"]
    t_url = tournament["url"]
    
    print(f"\n{'='*70}")
    print(f"📋 {t_name}")
    print(f"{'='*70}")
    
    # Step 1: Fetch tournament page
    print(f"Fetching schedule...")
    soup = fetch_page(t_url, delay=False)
    if not soup:
        print(f"  ❌ Failed to fetch tournament page")
        continue
    
    # Step 2: Discover matches
    score_links = soup.find_all('a', class_='ergebnis-link')
    matches = []
    
    for link in score_links:
        match_id = link.get('id')
        if not match_id or not match_id.isdigit():
            continue
        if any(m['match_id'] == match_id for m in matches):
            continue
        
        row = link.find_parent('tr')
        if not row:
            continue
        
        team_links = row.find_all('a', href=re.compile(r'/startseite/verein/\d+'))
        teams = []
        for tl in team_links:
            title = tl.get('title', '')
            if title and title not in teams:
                teams.append(title)
        
        if len(teams) >= 2:
            matches.append({
                'match_id': match_id,
                'team1': teams[0],
                'team2': teams[1],
            })
    
    print(f"  Found {len(matches)} matches")
    
    if not matches:
        print(f"  ⚠️ No matches found, skipping tournament")
        continue
    
    # Step 3: Scrape each match
    output_dir = f"../quizzes/starting11/{t_code}"
    os.makedirs(output_dir, exist_ok=True)
    
    success = 0
    skipped = 0
    failed = []
    
    for i, match in enumerate(matches):
        match_id = match['match_id']
        team1 = match['team1']
        team2 = match['team2']
        
        file1 = f"match_{match_id}_{team1.lower().replace(' ', '_')}.json"
        file2 = f"match_{match_id}_{team2.lower().replace(' ', '_')}.json"
        
        # Check if exists
        if SKIP_EXISTING and os.path.exists(os.path.join(output_dir, file1)):
            skipped += 1
            continue
        
        print(f"  [{i+1}/{len(matches)}] {team1} vs {team2}...", end=" ", flush=True)
        
        lineup_url = f"/spielbericht/aufstellung/spielbericht/{match_id}"
        index_url = f"/spielbericht/index/spielbericht/{match_id}"
        
        try:
            # Scrape lineup
            lineup_data = scrape_lineup_page(lineup_url)
            if len(lineup_data) < 22:
                print(f"❌ Not enough players ({len(lineup_data)})")
                failed.append((team1, team2, t_name, f"Not enough players ({len(lineup_data)})"))
                continue
            
            # Scrape positions
            positions_data = scrape_positions_page(index_url)
            if len(positions_data) < 2:
                print(f"❌ Missing positions")
                failed.append((team1, team2, t_name, "Missing positions"))
                continue
            
            # Merge and create quizzes
            team1_players = lineup_data[:11]
            team2_players = lineup_data[11:22]
            
            team1_merged = merge_player_data(team1_players, positions_data[0])
            team2_merged = merge_player_data(team2_players, positions_data[1])
            
            # Create quiz JSON
            quiz1 = create_quiz(team1, team1[:3].upper(), team2, team1_merged, t_name, "Match", "")
            quiz2 = create_quiz(team2, team2[:3].upper(), team1, team2_merged, t_name, "Match", "")
            
            # Save files
            with open(os.path.join(output_dir, file1), 'w') as f:
                json.dump(quiz1, f, indent=2)
            with open(os.path.join(output_dir, file2), 'w') as f:
                json.dump(quiz2, f, indent=2)
            
            print(f"✅")
            success += 1
            all_countries.add(team1)
            all_countries.add(team2)
            
        except Exception as e:
            print(f"❌ Error: {e}")
            failed.append((team1, team2, t_name, str(e)))
    
    # Tournament summary
    print(f"\n  📊 {t_name} Summary:")
    print(f"     ✅ Scraped: {success} matches ({success * 2} quizzes)")
    print(f"     ⏭️ Skipped: {skipped} matches")
    print(f"     ❌ Failed: {len(failed)} matches")
    
    grand_total_success += success
    grand_total_skipped += skipped
    grand_total_failed += len(failed)
    all_failed_matches.extend(failed)

# Final summary
print("\n" + "=" * 70)
print("🏆 GRAND TOTAL")
print("=" * 70)
print(f"✅ Total scraped: {grand_total_success} matches ({grand_total_success * 2} quizzes)")
print(f"⏭️ Total skipped: {grand_total_skipped} matches")
print(f"❌ Total failed: {grand_total_failed} matches")
print(f"\n🌍 Unique countries: {len(all_countries)}")
print(sorted(all_countries))

if all_failed_matches:
    print(f"\n❌ All failed matches:")
    for t1, t2, tourn, reason in all_failed_matches:
        print(f"   [{tourn}] {t1} vs {t2}: {reason}")

In [ ]:
# Cell: Extract unique countries from all scraped quizzes
# Run this after scraping to get the definitive country list

import glob

quiz_base = "../quizzes/starting11"
all_countries = set()

for tournament_dir in ["wc2022", "wc2018", "wc2014", "wc2010", "wc2006", "wc2002", "wc1998"]:
    path = os.path.join(quiz_base, tournament_dir, "*.json")
    files = glob.glob(path)
    
    for filepath in files:
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                country = data.get("answer", {}).get("country", "")
                if country:
                    all_countries.add(country)
        except:
            pass
    
    print(f"{tournament_dir}: {len(files)} files")

print(f"\n{'='*60}")
print(f"UNIQUE COUNTRIES ({len(all_countries)} total):")
print("="*60)

# Print as Python list for copy-paste
print("\nCOUNTRIES = [")
for country in sorted(all_countries):
    print(f'    "{country}",')
print("]")

# Starting11 Scraper

Scrapes World Cup match lineups from Transfermarkt to generate quiz data.

**What it extracts:**
- Starting 11 players with names and IDs
- Club affiliations at match time (with badge URLs)
- Pitch positions (x, y coordinates from formation diagram)

**Usage:**
1. Run all setup cells (1-4)
2. Set your match URL in the "Scrape Match" section
3. Run the scraper and generator cells
4. Quiz JSON and HTML preview will be created

In [19]:
# Cell 1: Setup and imports
import requests
from bs4 import BeautifulSoup
import re
import json
import os
import time

REQUEST_DELAY = 3  # Seconds between requests - respect rate limits!
BASE_URL = "https://www.transfermarkt.com"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

def fetch_page(url, delay=True):
    """Fetch a page with rate limiting and proper headers."""
    if delay:
        print(f"  Waiting {REQUEST_DELAY}s...")
        time.sleep(REQUEST_DELAY)
    
    full_url = url if url.startswith('http') else BASE_URL + url
    print(f"  Fetching: {full_url}")
    
    try:
        response = requests.get(full_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except Exception as e:
        print(f"  Error: {e}")
        return None

print("✅ Setup complete")

✅ Setup complete


In [20]:
# Cell 2: Scraper functions

def scrape_lineup_page(lineup_url):
    """
    Scrape player details from the lineup page (/aufstellung/).
    Returns list of players with name, ID, club info.
    """
    soup = fetch_page(lineup_url, delay=True)
    if not soup:
        return []
    
    players = []
    inline_tables = soup.find_all('table', class_='inline-table')
    
    for inline_table in inline_tables:
        outer_td = inline_table.find_parent('td')
        if not outer_td:
            continue
        outer_row = outer_td.find_parent('tr')
        if not outer_row:
            continue
            
        cells = outer_row.find_all('td', recursive=False)
        if len(cells) < 4:
            continue
        
        # Cell 0: Jersey number
        jersey_text = cells[0].get_text(strip=True)
        jersey = int(jersey_text) if jersey_text.isdigit() else None
        
        # Cell 1: Player info
        player_link = cells[1].find('a', href=re.compile(r'/profil/spieler/\d+'))
        if not player_link:
            continue
            
        href = player_link.get('href', '')
        player_match = re.search(r'/([^/]+)/profil/spieler/(\d+)', href)
        if not player_match:
            continue
            
        player_slug = player_match.group(1)
        player_id = player_match.group(2)
        
        name_link = cells[1].find('a', href=re.compile(r'/leistungsdatendetails/'))
        player_name = name_link.get('title', '') if name_link else player_slug.replace('-', ' ').title()
        
        # Cell 3: Club info
        club_link = cells[3].find('a', href=re.compile(r'/startseite/verein/\d+'))
        if not club_link:
            continue
            
        club_href = club_link.get('href', '')
        club_match = re.search(r'/([^/]+)/startseite/verein/(\d+)', club_href)
        if not club_match:
            continue
            
        club_id = club_match.group(2)
        club_name = club_link.get('title', '') or club_match.group(1).replace('-', ' ').title()
        
        players.append({
            'jersey': jersey,
            'name': player_name,
            'player_id': player_id,
            'club_name': club_name,
            'club_id': club_id
        })
    
    return players


def scrape_positions_page(index_url):
    """
    Scrape player positions from the index page (/index/).
    Returns list of teams, each with players and their x,y coordinates.
    """
    soup = fetch_page(index_url, delay=True)
    if not soup:
        return []
    
    teams = []
    lineup_boxes = soup.find_all('div', class_=re.compile(r'large-6.*columns|columns.*large-6'))
    
    for box in lineup_boxes:
        team_players = []
        all_divs = box.find_all('div', style=True)
        
        for elem in all_divs:
            style = elem.get('style', '')
            text = elem.get_text(strip=True)
            
            if not text or not re.match(r'^\d+\D', text):
                continue
            
            # Handle decimal percentages (e.g., 52.5%)
            top_match = re.search(r'top\s*:\s*([\d.]+)%', style)
            left_match = re.search(r'left\s*:\s*([\d.]+)%', style)
            
            if top_match and left_match:
                top_pct = float(top_match.group(1))
                left_pct = float(left_match.group(1))
                
                jersey_match = re.match(r'(\d+)', text)
                if jersey_match:
                    jersey = int(jersey_match.group(1))
                    name = text[len(jersey_match.group(1)):].strip()
                    
                    if not any(p['jersey'] == jersey for p in team_players):
                        team_players.append({
                            'jersey': jersey,
                            'name': name,
                            'x': left_pct,
                            'y': top_pct
                        })
        
        if len(team_players) == 11:
            teams.append(team_players)
    
    return teams

print("✅ Scraper functions defined")

✅ Scraper functions defined


In [21]:
# Cell 3: Quiz generation functions

def merge_player_data(players_with_clubs, positions):
    """Merge player club data with position data by jersey number."""
    pos_by_jersey = {p['jersey']: p for p in positions}
    merged = []
    
    # First, calculate the center offset to center the formation horizontally
    x_values = [p['x'] for p in positions]
    avg_x = sum(x_values) / len(x_values) if x_values else 50
    x_offset = 50 - avg_x  # Shift to center
    
    # Scale positions to add padding (map 0-100 to 15-85 for centered look)
    def scale_pos(val, padding=15):
        return padding + (val * (100 - 2 * padding) / 100)
    
    for player in players_with_clubs:
        jersey = player['jersey']
        pos = pos_by_jersey.get(jersey, {})
        
        # Center horizontally, then scale both axes
        raw_x = pos.get('x', 50) + x_offset
        raw_y = pos.get('y', 50)
        
        merged.append({
            'jersey': jersey,
            'name': player['name'],
            'player_id': player['player_id'],
            'club': {
                'name': player['club_name'],
                'club_id': player['club_id'],
                'badge_url': f"https://tmssl.akamaized.net/images/wappen/head/{player['club_id']}.png"
            },
            'position': {
                'x': scale_pos(raw_x),
                'y': scale_pos(raw_y)
            }
        })
    
    return merged


def create_quiz(players, team_name, country_code, match_info):
    """Create a Starting11 quiz JSON structure."""
    return {
        'answer': {
            'country': team_name,
            'country_code': country_code
        },
        'match': match_info,
        'players': players
    }

print("✅ Quiz functions defined")

✅ Quiz functions defined


In [22]:
# Cell 4: HTML visualization generator

def generate_html(quiz, output_path):
    """Generate an HTML visualization of the quiz."""
    match = quiz['match']
    answer = quiz['answer']['country']
    
    html = f'''<!DOCTYPE html>
<html>
<head>
    <title>Starting11 - {match['tournament']}</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: -apple-system, BlinkMacSystemFont, sans-serif; background: #1a1a2e; color: white; }}
        .container {{ max-width: 900px; margin: 0 auto; padding: 20px; }}
        h1 {{ text-align: center; margin-bottom: 10px; }}
        .subtitle {{ text-align: center; color: #888; margin-bottom: 30px; }}
        .pitch {{
            position: relative; width: 100%; aspect-ratio: 1.4;
            background: linear-gradient(to bottom, #2d5a27 0%, #3d7a37 50%, #2d5a27 100%);
            border-radius: 8px; border: 3px solid rgba(255,255,255,0.8); overflow: hidden;
        }}
        .pitch::before {{
            content: ''; position: absolute; top: 50%; left: 50%;
            transform: translate(-50%, -50%); width: 100px; height: 100px;
            border: 2px solid rgba(255,255,255,0.4); border-radius: 50%;
        }}
        .pitch::after {{
            content: ''; position: absolute; top: 50%; left: 0; right: 0;
            height: 2px; background: rgba(255,255,255,0.4);
        }}
        .penalty-area {{ position: absolute; left: 50%; transform: translateX(-50%); width: 44%; height: 16%; border: 2px solid rgba(255,255,255,0.4); }}
        .penalty-top {{ top: 0; border-top: none; }}
        .penalty-bottom {{ bottom: 0; border-bottom: none; }}
        .player {{
            position: absolute; display: flex; flex-direction: column; align-items: center;
            transform: translate(-50%, -50%); transition: transform 0.2s; cursor: pointer;
        }}
        .player:hover {{ transform: translate(-50%, -50%) scale(1.15); z-index: 10; }}
        .badge {{ width: 50px; height: 50px; background: white; border-radius: 8px; padding: 3px; box-shadow: 0 2px 10px rgba(0,0,0,0.5); }}
        .badge img {{ width: 100%; height: 100%; object-fit: contain; }}
        .jersey {{ margin-top: 4px; background: rgba(0,0,0,0.85); padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: bold; }}
        .answer-box {{ margin-top: 30px; text-align: center; }}
        .answer-box input {{ padding: 12px 20px; font-size: 18px; border: none; border-radius: 8px; width: 300px; }}
        .answer-box button {{ padding: 12px 30px; font-size: 18px; background: #4CAF50; color: white; border: none; border-radius: 8px; cursor: pointer; margin-left: 10px; }}
        .answer-box button:hover {{ background: #45a049; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>⚽ Starting 11</h1>
        <p class="subtitle">{match['tournament']} {match['stage']} • Guess the National Team</p>
        <div class="pitch">
            <div class="penalty-area penalty-top"></div>
            <div class="penalty-area penalty-bottom"></div>
'''
    
    for p in quiz['players']:
        x = p['position']['x']
        y = p['position']['y']
        badge = p['club']['badge_url']
        club = p['club']['name']
        html += f'''            <div class="player" style="left: {x}%; top: {y}%;" title="{club}">
                <div class="badge"><img src="{badge}" alt="{club}"></div>
                <span class="jersey">#{p['jersey']}</span>
            </div>
'''
    
    html += f'''        </div>
        <div class="answer-box">
            <input type="text" placeholder="Which national team?" id="guess">
            <button onclick="check()">Submit</button>
        </div>
    </div>
    <script>
        function check() {{
            const g = document.getElementById('guess').value.toLowerCase();
            if (g.includes('{answer.lower()[:5]}')) alert('🎉 Correct! {answer}!');
            else alert('❌ Try again!');
        }}
        document.getElementById('guess').addEventListener('keypress', e => {{ if (e.key === 'Enter') check(); }});
    </script>
</body>
</html>'''
    
    with open(output_path, 'w') as f:
        f.write(html)
    print(f"✅ Saved: {output_path}")

print("✅ HTML generator defined")

✅ HTML generator defined


---
## Scrape a Match

Set the match URLs below and run the cells.

In [23]:
# Cell 5: Configure match to scrape

# Match URLs (get these from Transfermarkt)
LINEUP_URL = "https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879"
INDEX_URL = "https://www.transfermarkt.com/argentina_france/index/spielbericht/3975879"

# Match metadata
MATCH_INFO = {
    "tournament": "World Cup 2022",
    "stage": "Final",
    "date": "2022-12-18",
    "venue": "Lusail Stadium"
}

# Team info (order matches how they appear on the page - home team first)
TEAM_1 = {"name": "Argentina", "code": "ARG", "opponent": "France"}
TEAM_2 = {"name": "France", "code": "FRA", "opponent": "Argentina"}

print(f"Configured: {TEAM_1['name']} vs {TEAM_2['name']}")
print(f"Tournament: {MATCH_INFO['tournament']} {MATCH_INFO['stage']}")

Configured: Argentina vs France
Tournament: World Cup 2022 Final


In [24]:
# Cell 6: Scrape the match data

print("=" * 50)
print("SCRAPING MATCH DATA")
print("=" * 50)

# Step 1: Get player details (names, IDs, clubs)
print("\n[1/2] Scraping lineup page for player details...")
all_players = scrape_lineup_page(LINEUP_URL)
print(f"  → Found {len(all_players)} players")

# Step 2: Get positions from formation diagram
print("\n[2/2] Scraping index page for positions...")
all_positions = scrape_positions_page(INDEX_URL)
print(f"  → Found positions for {len(all_positions)} teams")

# Split into teams
team1_players = all_players[:11]
team2_players = all_players[11:22]
team1_positions = all_positions[0] if len(all_positions) > 0 else []
team2_positions = all_positions[1] if len(all_positions) > 1 else []

print("\n" + "=" * 50)
print("SCRAPING COMPLETE")
print("=" * 50)

SCRAPING MATCH DATA

[1/2] Scraping lineup page for player details...
  Waiting 3s...
  Fetching: https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879
  → Found 50 players

[2/2] Scraping index page for positions...
  Waiting 3s...
  Fetching: https://www.transfermarkt.com/argentina_france/index/spielbericht/3975879
  → Found positions for 2 teams

SCRAPING COMPLETE


In [25]:
# Cell 7: Merge and preview data

# Merge club info with positions
team1_merged = merge_player_data(team1_players, team1_positions)
team2_merged = merge_player_data(team2_players, team2_positions)

print(f"\n{TEAM_1['name']} Starting XI:")
print("-" * 70)
for p in team1_merged:
    print(f"  #{p['jersey']:2}  {p['name']:<22}  {p['club']['name']:<25}  ({p['position']['x']:5.1f}, {p['position']['y']:5.1f})")

print(f"\n{TEAM_2['name']} Starting XI:")
print("-" * 70)
for p in team2_merged:
    print(f"  #{p['jersey']:2}  {p['name']:<22}  {p['club']['name']:<25}  ({p['position']['x']:5.1f}, {p['position']['y']:5.1f})")


Argentina Starting XI:
----------------------------------------------------------------------
  #23  Emiliano Martínez       Aston Villa                ( 49.9,  71.0)
  #13  Cristian Romero         Tottenham Hotspur          ( 58.7,  59.1)
  #19  Nicolás Otamendi        SL Benfica                 ( 41.5,  59.1)
  # 3  Nicolás Tagliafico      Olympique Lyon             ( 27.2,  57.7)
  #26  Nahuel Molina           Atlético de Madrid         ( 73.0,  57.7)
  #24  Enzo Fernández          SL Benfica                 ( 49.9,  42.3)
  # 7  Rodrigo De Paul         Atlético de Madrid         ( 59.0,  34.6)
  #20  Alexis Mac Allister     Brighton & Hove Albion     ( 40.8,  34.6)
  #11  Ángel Di María          Juventus FC                ( 32.4,  22.0)
  #10  Lionel Messi            Paris Saint-Germain        ( 67.4,  22.0)
  # 9  Julián Alvarez          Manchester City            ( 49.9,  17.1)

France Starting XI:
----------------------------------------------------------------------
  # 1  Hug

In [26]:
# Cell 8: Generate quizzes and save

# Create quiz structures
quiz_team1 = create_quiz(
    team1_merged,
    TEAM_1['name'],
    TEAM_1['code'],
    {**MATCH_INFO, 'opponent': TEAM_1['opponent']}
)

quiz_team2 = create_quiz(
    team2_merged,
    TEAM_2['name'],
    TEAM_2['code'],
    {**MATCH_INFO, 'opponent': TEAM_2['opponent']}
)

# Save JSON files
output_dir = '../quizzes/starting11/wc2022'
os.makedirs(output_dir, exist_ok=True)

team1_filename = f"{MATCH_INFO['stage'].lower()}_{TEAM_1['name'].lower()}.json"
team2_filename = f"{MATCH_INFO['stage'].lower()}_{TEAM_2['name'].lower()}.json"

with open(f"{output_dir}/{team1_filename}", 'w') as f:
    json.dump(quiz_team1, f, indent=2)
print(f"✅ Saved: {output_dir}/{team1_filename}")

with open(f"{output_dir}/{team2_filename}", 'w') as f:
    json.dump(quiz_team2, f, indent=2)
print(f"✅ Saved: {output_dir}/{team2_filename}")

✅ Saved: ../quizzes/starting11/wc2022/final_argentina.json
✅ Saved: ../quizzes/starting11/wc2022/final_france.json


In [27]:
# Cell 9: Generate HTML previews

generate_html(quiz_team1, f"starting11_{TEAM_1['name'].lower()}.html")
generate_html(quiz_team2, f"starting11_{TEAM_2['name'].lower()}.html")

print("\n🎉 Done! Open the HTML files in a browser to preview.")

✅ Saved: starting11_argentina.html
✅ Saved: starting11_france.html

🎉 Done! Open the HTML files in a browser to preview.
